**Data Quality Check — Summary**

Both tables are clean. No fixes or exclusions needed before the A/B analysis.

| Check | Result |
|---|---|
| NULLs | None anywhere, except `conversion_date` (97.23% of players) |
| Table sizes | `activity`: 214,878,701 rows · `assignment`: 10,331,056 rows |
| `playerid` uniqueness | 1 row per player in `assignment`; 1 row per player+day in `activity` |
| Date logic | `install_date` ≤ `assignment_date` and ≤ `conversion_date` for all players — 0 violations |
| Date ranges | installs 2016-01-01 → 2017-05-22 · assignment 2017-05-04 → 2017-05-22 |
| `purchases` | 0–514, 98.79% zero, mean 0.03, p99 = 1 |
| `gameends` | 0–143, 0.02% zero, mean 13.15, median 10, p99 = 59 |

- **NULL `conversion_date` is expected, not missing data** — it flags players who had not made a first purchase before assignment.
- **One player = one install date, one conversion date, one group, one assignment date.** No duplicate or conflicting assignment rows.
- **`purchases` follows a standard whale distribution:** a small payer base, and inside it a tiny fraction driving most of the volume. Median/IQR are 0 and uninformative — use means and tail cuts, not quantiles.
- **No purchases recorded on rows with `gameends = 0`**, which suggests activity rows are session-driven.
- Assignment window (2017-05-04 → 2017-05-22) matches the provided A/B test info.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from bigquery import run_query

# Data Quality Check

- Check NULL data --- Look good
- Check extreme values (no. rounds and purchases, date range) -- Look good.
- 1 player can have only a install date, conversion date, and belong to one group --- YES
- install date <= conversation date and assignment date -- Lood good
- uniqueness of playerid+activity_date in the avitity datatable

## Null values

- Null for conversion_date it means users are non-paying users until the day they are assigned into AB test. 
- Asignment table. Only conversion_date as expected
- Activity table. No NULL values

In [3]:
activity_null_sql = """
SELECT
  COUNT(*) AS n_rows,
  COUNTIF(playerid IS NULL) AS null_playerid,
  COUNTIF(activity_date IS NULL) AS null_activity_date,
  COUNTIF(purchases IS NULL) AS null_purchases,
  COUNTIF(gameends IS NULL) AS null_gameends,
  ROUND(100 * COUNTIF(purchases IS NULL) / COUNT(*), 2) AS pct_null_purchases,
  ROUND(100 * COUNTIF(gameends  IS NULL) / COUNT(*), 2) AS pct_null_gameends
FROM `king-ds-recruit-candidate-1114.abtest.activity`
"""
activity_null_df = run_query(activity_null_sql)
print(activity_null_df)

      n_rows  null_playerid  null_activity_date  null_purchases  \
0  214878701              0                   0               0   

   null_gameends  pct_null_purchases  pct_null_gameends  
0              0                 0.0                0.0  


In [4]:
assignment_null_sql = """
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT playerid) AS n_distinct_players,
  COUNTIF(playerid IS NULL) AS null_playerid,
  COUNTIF(abtest_group IS NULL) AS null_group,
  COUNTIF(TRIM(abtest_group) = '') AS empty_group,
  COUNTIF(assignment_date IS NULL) AS null_assignment_date,
  COUNTIF(install_date IS NULL) AS null_install_date,
  COUNTIF(conversion_date IS NULL) AS null_conversion_date,
  ROUND(100 * COUNTIF(conversion_date IS NULL) / COUNT(*), 2) AS pct_never_converted
FROM `king-ds-recruit-candidate-1114.abtest.assignment`
"""
assignment_null_df = run_query(assignment_null_sql)
print(assignment_null_df)

     n_rows  n_distinct_players  null_playerid  null_group  empty_group  \
0  10331056            10331056              0           0            0   

   null_assignment_date  null_install_date  null_conversion_date  \
0                     0                  0              10045133   

   pct_never_converted  
0                97.23  


# Extreme values

In [ ]:
# Assignment table: Date data for assignment data. Look normal and consistent with the provided ab test info

assignment_extreme_sql = """
SELECT 
  MIN(install_date) AS first_install_date, 
  MAX(install_date) AS last_install_date,
  MIN(conversion_date) AS min_first_purchase_date, 
  MAX(conversion_date) AS max_first_purchase_date,
  MIN(assignment_date) AS first_ab_test_date,
  MAX(assignment_date) AS last_ab_test_date
FROM `king-ds-recruit-candidate-1114.abtest.assignment`
"""
assignment_extreme_df = run_query(assignment_extreme_sql)
print(assignment_extreme_df)

  first_install_date last_install_date min_first_purchase_date  \
0         2016-01-01        2017-05-22              2016-01-01   

  max_first_purchase_date first_ab_test_date last_ab_test_date  
0              2017-05-22         2017-05-04        2017-05-22  


In [8]:
# Number of purchase range: GOOD. Extreme right tail. 
# Standard whale distribution: a small payer base, and within it a tiny fraction driving most of the volume.
# Median and IQR are meaningless here

purchase_range_sql = """
SELECT
  COUNT(*)                                          AS n_rows,
  COUNTIF(purchases IS NULL)                        AS n_null,
  COUNTIF(purchases < 0)                            AS n_negative,
  COUNTIF(purchases = 0)                            AS n_zero,
  ROUND(100 * COUNTIF(purchases = 0) / COUNT(*), 2) AS pct_zero,
  MIN(purchases)                                    AS min_purchase,
  MAX(purchases)                                    AS max_purchase,
  ROUND(AVG(purchases), 2)                          AS mean_purchase,
  ROUND(STDDEV(purchases), 2)                       AS sd_purchase,
  APPROX_QUANTILES(purchases, 100)[OFFSET(25)]      AS p25,
  APPROX_QUANTILES(purchases, 100)[OFFSET(50)]      AS median_purchase,
  APPROX_QUANTILES(purchases, 100)[OFFSET(75)]      AS p75,
  APPROX_QUANTILES(purchases, 100)[OFFSET(75)]
    - APPROX_QUANTILES(purchases, 100)[OFFSET(25)]  AS iqr,
  APPROX_QUANTILES(purchases, 100)[OFFSET(90)]      AS p90,
  APPROX_QUANTILES(purchases, 100)[OFFSET(99)]      AS p99
FROM `king-ds-recruit-candidate-1114.abtest.activity`
"""
purchase_range_df = run_query(purchase_range_sql)
print(purchase_range_df)

      n_rows  n_null  n_negative     n_zero  pct_zero  min_purchase  \
0  214878701       0           0  212270624     98.79             0   

   max_purchase  mean_purchase  sd_purchase  p25  median_purchase  p75  iqr  \
0           514           0.03         0.77    0                0    0    0   

   p90  p99  
0    0    1  


In [9]:
# double check the whale distribution by looking at top payers
purchase_top_sql = """
SELECT
  *
FROM `king-ds-recruit-candidate-1114.abtest.activity`
WHERE purchases > 1
ORDER BY purchases DESC, playerid, activity_date
LIMIT 1000
"""
purchase_top_df = run_query(purchase_top_sql)
print(purchase_top_df)

     playerid activity_date  purchases  gameends
0    23160565    2017-04-22        514        75
1    41458856    2017-05-15        504        53
2    41458856    2017-04-28        462        54
3    32692136    2017-05-13        461        83
4    41458856    2017-04-21        457        54
..        ...           ...        ...       ...
995  20130671    2017-05-21        132       105
996  22273649    2017-05-19        132        27
997  37342489    2017-04-22        132        42
998  37437312    2017-04-22        132        20
999  37654460    2017-04-22        132        63

[1000 rows x 4 columns]


In [10]:
# Gameends schema and range. GOOD. It seems the activity is recorded based on sessions.
gamerounds_range_sql = """
SELECT
  COUNT(*) AS n_rows,
  COUNTIF(gameends IS NULL) AS n_null,
  COUNTIF(gameends < 0) AS n_negative,
  COUNTIF(gameends = 0) AS n_zero,
  ROUND(100 * COUNTIF(gameends = 0) / COUNT(*), 2) AS pct_zero,
  MIN(gameends) AS min_rounds,
  MAX(gameends) AS max_rounds,
  ROUND(AVG(gameends), 2) AS mean_rounds,
  ROUND(STDDEV(gameends), 2) AS sd_rounds,
  APPROX_QUANTILES(gameends, 100)[OFFSET(25)] AS p25,
  APPROX_QUANTILES(gameends, 100)[OFFSET(50)] AS median_rounds,
  APPROX_QUANTILES(gameends, 100)[OFFSET(75)] AS p75,
  APPROX_QUANTILES(gameends, 100)[OFFSET(75)]
    - APPROX_QUANTILES(gameends, 100)[OFFSET(25)] AS iqr,
  APPROX_QUANTILES(gameends, 100)[OFFSET(90)] AS p90,
  APPROX_QUANTILES(gameends, 100)[OFFSET(99)] AS p99
FROM `king-ds-recruit-candidate-1114.abtest.activity`
"""
gamerounds_range_df = run_query(gamerounds_range_sql)
print(gamerounds_range_df)

      n_rows  n_null  n_negative  n_zero  pct_zero  min_rounds  max_rounds  \
0  214878701       0           0   38431      0.02           0         143   

   mean_rounds  sd_rounds  p25  median_rounds  p75  iqr  p90  p99  
0        13.15      10.22    8             10   15    7   23   59  


In [ ]:
# activity table is triggered by session based? check purchase with 0 round = 0. It seems the activity is recorded based on sessions.
gamerounds_purchase_sql = """
SELECT
  COUNTIF(purchases > 0 AND gameends = 0) AS purchase_without_round,
FROM `king-ds-recruit-candidate-1114.abtest.activity`
"""
gamerounds_purchase_df = run_query(gamerounds_purchase_sql)
print(gamerounds_purchase_df)

   purchase_without_round
0                       0


# Uniqueness of playerid

In [ ]:
# 1 player can have only a install date, conversion date, and belong to one group: YES
date_sql = """
SELECT 
  playerid,
  count(distinct conversion_date) number_first_purchase_date, 
  count(distinct install_date) number_first_install_date,
  count(distinct abtest_group) number_groups,
  count(distinct assignment_date) number_abtest_date
FROM `king-ds-recruit-candidate-1114.abtest.assignment` 
GROUP BY all
having count(distinct conversion_date) > 1 
        or count(distinct install_date)  > 1
        or count(distinct abtest_group)  > 1
        or count(distinct assignment_date) > 1
"""
date_df = run_query(date_sql)
print(date_df)

Empty DataFrame
Columns: [playerid, number_first_purchase_date, number_first_install_date, number_groups, number_abtest_date]
Index: []


In [ ]:
# install date <= conversation date and install_date <= assignment date: YES
date_valid_sql = """
SELECT 
  count(*)
FROM `king-ds-recruit-candidate-1114.abtest.assignment` 
Where 
  install_date > assignment_date
  or (
    conversion_date is not null
    and install_date > conversion_date
  )
"""
date_valid_df = run_query(date_valid_sql)
print(date_valid_df)

   f0_
0    0


In [12]:
# uniqueness of playerid+activity_date in the avitity datatable
# quality: user_id per row per day - GOOD
id_unique_sql = """
SELECT 
  activity_date,
  playerid,
  count(*) no_rows --expect one user per day one row
FROM `king-ds-recruit-candidate-1114.abtest.activity` 
GROUP BY all
  having count(*) > 1
"""
id_unique_df = run_query(id_unique_sql)
print(id_unique_df)

Empty DataFrame
Columns: [activity_date, playerid, no_rows]
Index: []
